# BEA 3-way generator

An attempt at tidying the code up at least a little so that eg. Aisling has a fighting chance of seeing what it's even doing.

**Convert the raw cells to Python to check the functions are working OK**

The core aspect is the call to `amati_bea.induce_theory`, and we can then write the theory to a file. The rules will come from `clingo_program_files/bea_rules.amati`.

There's a number of legacy imports here.

In [1]:
import amati_bea as amati

In [2]:
import pandas as pd

from sklearn.metrics import f1_score

In [3]:
from operator import itemgetter
import pickle
import datetime
import os

## Import the data

We've got the DataFrames containing the data in the file `bea_3way_data.pickle`. Load that in:

## Run on a single example case as a tester

I realised that my code doesn't always assign the same Amati codes to the original IDs, so start by selecting a test case from the original question code:

Let's try working through the functions in `amati_bea.py`.

In [4]:
TEST_QUESTION_ID = "1f3469d5-f6a8-49f4-85d0-d2e156e4cb0c"
TEST_QUESTION_ID

'1f3469d5-f6a8-49f4-85d0-d2e156e4cb0c'

We now pull out the Series containing this ID as the one to work with:

For the responses, want just the training cases, not the trial cases.

OK, that should cover what we want.

----------------

## Generate the positive and negative cases

OK, now to start with the `amati_bea` functions.

Let's start off by generating positive and negative cases.

------------------------

I think what I want is to be able to send those structures to `amati_bea.py` and see how it goes from there.

Check that the function `build_amati_training_file` works correctly:

OK, that seems OK. Now, what about an actual induction run? Check the behaviour of `induce_once` first:

OK, and then do a complete run:

Actually, let's just call that the theory. Should be able to pickle it...

In [5]:
def induce_theory(
    original_question_id,
    data_file="bea_3way_data.pickle",
    rulesfile="clingo_program_files/bea_rules.amati",
    time_limit=120,
    min_precision=0.8,
    min_coverage=2,
):
    """question_id is the original version, not the"
    shortened version"""

    with open("bea_3way_data.pickle", "br") as fIn:
        d = pickle.load(fIn)
        all_questions_df = d["questions"]
        all_responses_df = d["responses"]
        all_text_df = d["text"]

    question_ss = all_questions_df.set_index("original_question_id").T[
        original_question_id
    ]
    question_id = question_ss["question_id"]

    # To induce the theory, just want the training cases

    responses_df = all_responses_df.query("`question_id`==@question_id").query(
        "`partition`=='train'"
    )

    texts_used = [
        question_ss["question_id"],
        question_ss["correct_id"],
        question_ss["partially_correct_id"],
        question_ss["incorrect_id"],
    ] + list(responses_df["response_id"])

    text_df = all_text_df.query("`id`.isin(@texts_used)")[["id", "i", "lemma"]]

    # Next, need the positive and negative cases

    accuracy_classes = set(responses_df["score"])

    pos_cases_df = responses_df[["response_id", "question_id", "score"]].sort_values(
        "response_id", ignore_index=True
    )

    neg_cases_df = (
        pd.concat([responses_df.assign(classes=c) for c in accuracy_classes])
        .query("`score`!=`classes`")
        .drop(["score", "partition"], axis="columns")
        .rename({"classes": "score"}, axis="columns")[
            ["response_id", "question_id", "score"]
        ]
        .sort_values("response_id", ignore_index=True)
    )

    output = {
        "theory": amati.induce_theory(
            positive_examples_df=pos_cases_df,
            negative_examples_df=neg_cases_df,
            responses_df=responses_df,
            text_df=text_df,
            question_ss=question_ss,
            rulesfile=rulesfile,
            time_limit=time_limit,
            min_precision=min_precision,
            min_coverage=min_coverage,
        )
    }

    output["original_question_id"] = original_question_id
    output["question_id"] = question_id

    output["all_questions_df"] = all_questions_df
    output["all_responses_df"] = all_responses_df
    output["all_text_df"] = all_text_df

    with open(rulesfile) as fIn:
        output["rules"] = fIn.read()

    return output

## Apply to the complete set

We should now be able to apply to the complete set of questions. 

In [7]:
with open("bea_3way_data.pickle", "br") as fIn:
    p = pickle.load(fIn)

for q_id in set(p["questions"]["original_question_id"]):
    print(q_id)
    d = datetime.datetime.now()
    theory_file_name = os.path.join(
        "theories", "run_20260303", f"theory_{q_id}_{d.date()}_{d.time()}.pickle"
    )
    with open(theory_file_name, "wb") as fOut:
        pickle.dump(induce_theory(q_id), fOut)

75333f3a-c1c2-4729-8d61-146800f16532
145 128 110 99 88 82 76 71 67 63 58 54 52 48 46 43 41 39 37 35 33 31 29 27 25 23 52af038f-7137-4406-bcbb-fb5a0291fd25
116 107 101 97 93 90 85 83 81 79 77 75 73 71 69 67 64 62 60 57 55 53 51 49 e937d43d-23dd-49d7-a3c7-0f874e478b3c
40 36 31 28 26 24 22 20 0bf8fd8c-a68a-4491-ae28-be767f335729
40 13 6 faf4a6a7-85fd-4cb7-ab8e-1c21b4bbdec9
128 111 102 95 92 87 82 79 75 70 68 66 63 61 58 56 54 52 50 48 45 43 41 39 37 35 33 31 29 27 9f80f2c6-db27-4f50-b16b-e6f7b690d04a
45 35 32 29 26 22 20 18 16 14 c24f6093-2644-4fc4-b7c9-d798500f483f
153 147 143 138 134 130 127 124 121 118 116 113 111 109 107 105 102 100 98 96 94 92 90 88 86 84 82 80 78 76 74 72 81d2eaef-552d-4748-bad6-b38784de7303
59 53 50 47 45 43 41 39 72ec67ad-49f9-4aae-bd03-6efe95ec0947
38 29 25 21 19 17 15 90d2761e-010d-4463-afe7-6b3fa2561dc2
15 12 10 8 4e3a3e06-40c4-479b-8882-36005aae1e4e
23 15 13 11 9 7 b076d670-42c4-46fc-b4f4-07a2d617afd7
27 14 7 5 3 1f3469d5-f6a8-49f4-85d0-d2e156e4cb0c
216 183 16